# GenAI Pipeline — Testing Notebook

LLM-based screening of Fermentation-pillar patents for subpillar classification (BF/PF/NA).
Uses Claude with prompt caching via the Anthropic Python SDK.

### 1. Imports and Configuration

In [ ]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator
from typing import Literal
from pydantic import create_model

load_dotenv()

DB_PATH = "../../patents_training.db"
OUTPUT_DIR = Path(".")

### 2. Data Inspection

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df())

df_raw = con.sql("SELECT * FROM patents_raw").df()   # ← UPDATE: table name
con.close()

Tables:
                 name
0  patents_embeddings
1         patents_raw


In [7]:
df = df_raw[
    (df_raw["scope"] == "in") &
    (df_raw["pillar"].notna()) &
    (df_raw["subpillar"].notna()) 
].reset_index(drop=True)

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (scope=in, pillar not empty): {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPillar distribution:")
print(df["pillar"].value_counts())
print(f"\nSubpillar distribution:")
print(df["subpillar"].value_counts())
df.head()

Raw shape: (2450, 15)
Filtered shape (scope=in, pillar not empty): (138, 15)

Columns: ['id', 'family_id', 'application_number', 'title', 'abstract', 'cpc', 'publication_year', 'jurisdiction', 'scope', 'pillar', 'subpillar', 'research_category', 'endproduct', 'ingredient', 'truncated']

Pillar distribution:
pillar
F     136
CC      2
Name: count, dtype: int64

Subpillar distribution:
subpillar
BF    87
PF    51
Name: count, dtype: int64


,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,US-20230080653-A1,52673981,US17990954,MEAT SUBSTITUTE,"Described herein is an edible formulation, sui...","['A23L13/424', 'A23L33/185', 'A23J3/20', 'A23J...",2023,US,in,F,BF,End product formulation,Meat,NaN,False
1,US-20180014567-A1,52673981,US15546716,EDIBLE FUNGI,"An edible formulation, suitable for vegans, co...","['A23L33/175', 'A23L13/426', 'A23J3/20', 'A23L...",2018,US,in,F,BF,End product formulation,Meat,NaN,False
2,US-12612592-B2,63834283,US17273678,Method for enriching a biomass with proteins,The present invention relates to a method for ...,"['A23L2/66', 'A23J3/227', 'C12N1/125', 'A23L29...",2026,US,in,F,BF,Ingredient optimisation,Agnostic,NaN,False
3,US-20210340490-A1,63834283,US17273678,METHOD FOR ENRICHING A BIOMASS WITH PROTEINS,The present invention relates to a method for ...,"['A61K36/04', 'A23J1/008', 'A23J1/009', 'A23J3...",2021,US,in,F,BF,Ingredient optimisation,Agnostic,NaN,False
4,US-20250228265-A1,65011754,US19098773,FUNCTIONAL YEAST PROTEIN CONCENTRATE,The present invention relates to a method for ...,"['A23L31/10', 'A23L33/195', 'C12N1/18', 'A23V2...",2025,US,in,F,BF,Ingredient optimisation,Meat,"Isolates, concentrates, and flours",False


In [8]:
df_f  = df[df["pillar"] == "F"].reset_index(drop=True)

### 3. Balanced Subset Creation

Taking up to 30 patents from each subpillar (BF, PF) within the Fermentation pillar.


In [9]:
RANDOM_STATE=4

def create_balanced_sample(df, category_counts, random_state=RANDOM_STATE):
    """
    category_counts: dict mapping subpillar -> n
    Categories with no matching rows are skipped with a warning.
    If n exceeds available rows, all available rows are taken (with a warning).
    """
    samples = []
    for cat, n in category_counts.items():
        subset = df[df["subpillar"] == cat]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        if n > available:
            print(f"  Warning: '{cat}' — requested {n} but only {available} available, taking all.")
            n = available
        samples.append(subset.sample(n=n, random_state=random_state))
    combined = pd.concat(samples, ignore_index=True)
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [10]:
test_data_F = create_balanced_sample(df_f, {
    "BF": 30,
    "PF": 30,
}, random_state=RANDOM_STATE)
print(f"test_data_F: {test_data_F.shape}")
# test_data_F[["id", "title", "scope", "pillar"]]

test_data_F: (60, 15)


### 4. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.

In [12]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=OUTPUT_DIR):
    path = output_dir / filename
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(initial_test_data, "initial_test_data_rand3.xlsx")
save_subset(test_data_F, f"subpillar_test_data_F_rand{RANDOM_STATE}.csv")

Saved 60 records to subpillar_test_data_F_rand4.csv


### 5. Load Prompt and Select Dataset

In [22]:
AP_PILLAR = "F"  # subpillar only applies to Fermentation-pillar patents

# Read subset data from file
DATASET = pd.read_csv(f"subpillar_test_data_F_rand{RANDOM_STATE}.csv")
print(f"Pillar: {AP_PILLAR} | Dataset: {DATASET.shape[0]} records")

Pillar: F | Dataset: 60 records


In [28]:
PROMPT_VERSION = "v2"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"{PROMPT_VERSION}/prompt_subpillar_patents_{PROMPT_VERSION}.md"

In [30]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food technology.

This patent has already been classified as belonging to the Fermentation pillar (as opposed to Plant-based, which covers fermentation used only as a processing method for plant proteins, or Cultivated, which covers animal cell cultivation). Your task is to classify it into a fermentation subpillar based on its title and abstract.

Before assigning a subpillar, identify what the invention describes as its product: the microbial, fungal, or algal biomass itself, or a specific molecule produced by and purified away from that biomass.

Key decision rules:
- Mycoprotein, single-cell protein, or microbial/fungal/algal biomass composition, where the organism's cell mass itself is the food ingredient or product → BF
- Recombinant or heterologous expression, or engineered/selected production, of a specific named protein, enzyme, lipid, or other defined molecule (e.g. "a host cell engineered to express X", "recombinant production of

### 6. API Call with Prompt Caching

In [31]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

In [32]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

SUBPILLAR_CATS = ["BF", "PF", "NA"]

def make_schema(cats, include_reasoning):
    # The LLM sometimes returns lowercase/differently-cased values (e.g. "bf") despite the
    # enum constraint; cats_map allows case-insensitive lookup so the field_validator can
    # normalise the value before Pydantic validates it.
    cats_map = {c.lower(): c for c in cats}
    cat_type = Literal[*cats]

    class _Base(BaseModel):
        # check_fields=False because "subpillar" is added by create_model, not defined here
        @field_validator("subpillar", mode="before", check_fields=False)
        @classmethod
        def normalise_case(cls, v):
            if isinstance(v, str):
                return cats_map.get(v.lower(), v)
            return v

    fields = {"subpillar": (cat_type, ...)}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    return create_model("ClassificationSchema", __base__=_Base, **fields)

ClassificationSchema = make_schema(SUBPILLAR_CATS, INCLUDE_REASONING)
print(f"Schema built for subpillar classification: {SUBPILLAR_CATS}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_patent(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output

Schema built for subpillar classification: ['BF', 'PF', 'NA']


In [33]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


### 7. Error Handling with Retry

In [34]:
def classify_with_error_handling(row, system_prompt):
    patent_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_patent(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {patent_id}: model returned no structured output")
                return {"id": patent_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = patent_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {patent_id}: {last_error}")
    return {"id": patent_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers

In [35]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data

In [36]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Run 1 / 1
  [1/60] CN-119546745-A


  [2/60] US-20250136922-A1
  [3/60] WO-2025119806-A1
  [4/60] EP-4676238-A1
  [5/60] EP-4507513-A1
  [6/60] US-20260062667-A1
  [7/60] WO-2025158078-A1
  [8/60] EP-4476349-A2
  [9/60] CL-2026000428-A1
  [10/60] MX-2026003171-A
  [11/60] EP-4658754-A1
  [12/60] EP-4523544-A1
  [13/60] US-20250340825-A1
  [14/60] EP-3670646-A1
  [15/60] WO-2025114258-A1
  [16/60] US-20250027033-A1
  [17/60] EP-4626251-A1
  [18/60] WO-2025036973-A1
  [19/60] EP-4658753-A1
  [20/60] US-20250388852-A1
  [21/60] DE-102022201682-A1
  [22/60] US-12600940-B2
  [23/60] US-20250228265-A1
  [24/60] EP-4642904-A1
  [25/60] JP-2024501297-A
  [26/60] US-20250241331-A1
  [27/60] EP-4591714-A1
  [28/60] US-20250064095-A1
  [29/60] JP-2025517300-A
  [30/60] EP-4530341-A1
  [31/60] EP-4747269-A1
  [32/60] AR-130716-A1
  [33/60] JP-2026501609-A
  [34/60] WO-2025114239-A1
  [35/60] US-20250134149-A1
  [36/60] JP-2025533662-A
  [37/60] US-20240247227-A1
  [38/60] CN-120897670-A
  [39/60] WO-2025262697-A1
  [40/60] JP-202454

,subpillar_LLM,reasoning_LLM,id,status,run
0,BF,The invention concerns fungal biomass (mycelia...,CN-119546745-A,ok,1
1,BF,The patent describes a system for cultivating ...,US-20250136922-A1,ok,1
2,PF,The patent explicitly describes recombinant pr...,WO-2025119806-A1,ok,1
3,BF,The invention uses filamentous fungal biomass ...,EP-4676238-A1,ok,1
4,BF,The invention concerns extracting and processi...,EP-4507513-A1,ok,1
5,BF,The invention explicitly targets single-cell p...,US-20260062667-A1,ok,1
6,PF,The invention uses a recombinant protein (a pr...,WO-2025158078-A1,ok,1
7,PF,The invention explicitly uses precision fermen...,EP-4476349-A2,ok,1
8,BF,The invention is a yeast single-cell protein (...,CL-2026000428-A1,ok,1
9,PF,The patent explicitly concerns recombinant β-l...,MX-2026003171-A,ok,1


In [41]:
result_cols = ["id", "run", "subpillar_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "pillar", "subpillar"]].merge(
    results_df[result_cols], on="id", how="left"
)

comparison["correct"] = comparison["subpillar"] == comparison["subpillar_LLM"]

# Overall accuracy (single-label classification, no top-2 concept here)
n = len(comparison)
print(f"Accuracy: {comparison['correct'].mean():.0%}  (n={n})")

# Per-category breakdown
cat_stats = (
    comparison.groupby("subpillar")
    .agg(
        n=("correct", "count"),
        correct=("correct", "sum"),
    )
    .assign(
        accuracy=lambda d: (d["correct"] / d["n"]).map("{:.0%}".format),
    )
)
display(cat_stats)

# Confusion matrix: ground truth (rows) vs LLM prediction (columns) — more informative than
# a single accuracy number for a 3-way classification, shows exactly what's being confused with what
confusion = pd.crosstab(comparison["subpillar"], comparison["subpillar_LLM"])
display(confusion)

# Detail table
display_cols = ["id", "title", "abstract", "pillar", "subpillar", "subpillar_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["correct"]
comparison[display_cols]

Accuracy: 95%  (n=60)


,n,correct,accuracy
subpillar,,,
BF,30,29,97%
PF,30,28,93%


subpillar_LLM,BF,NA,PF
subpillar,,,
BF,29,1,0
PF,1,1,28


,id,title,abstract,pillar,subpillar,subpillar_LLM,reasoning_LLM,correct
0,CN-119546745-A,Reinforced fungal material,The present invention relates to reinforced fu...,F,BF,BF,The invention concerns fungal biomass (mycelia...,True
1,US-20250136922-A1,SYSTEM AND METHOD FOR CULTIVATING AND HARVESTI...,Disclosed is a system for cultivating and harv...,F,BF,BF,The patent describes a system for cultivating ...,True
2,WO-2025119806-A1,RECOMBINANT WILD TYPE PATATIN-2 AND MUTANTS TH...,The invention relates a recombinant wild type ...,F,PF,PF,The patent explicitly describes recombinant pr...,True
3,EP-4676238-A1,METHOD OF MAKING A FOOD PRODUCT USING MICROBIA...,Described is a method to prepare a food produc...,F,BF,BF,The invention uses filamentous fungal biomass ...,True
4,EP-4507513-A1,PROCESS FOR OBTAINING A TEXTURISING PROTEIN EX...,The present invention relates to a process for...,F,BF,BF,The invention concerns extracting and processi...,True
5,US-20260062667-A1,STRAINS AND PROCESSES FOR SINGLE CELL PROTEIN ...,A bacterial strain of the genus Xanthobacter a...,F,BF,BF,The invention explicitly targets single-cell p...,True
6,WO-2025158078-A1,ANIMAL-FREE SUBSTITUTE DAIRY FOOD PRODUCTS AND...,The invention relates to a method of producing...,F,PF,PF,The invention uses a recombinant protein (a pr...,True
7,EP-4476349-A2,METHODS FOR PRODUCTION OF ANIMAL-FREE HONEY AN...,"Royal jelly honey produced by nurse bees, hone...",F,PF,PF,The invention explicitly uses precision fermen...,True
8,CL-2026000428-A1,Single-cell protein products containing low le...,The present invention relates to a composition...,F,BF,BF,The invention is a yeast single-cell protein (...,True
9,MX-2026003171-A,MICROPARTICULATION OF RECOMBINANT BETA-LACTOGL...,The present invention relates to a method for ...,F,PF,PF,The patent explicitly concerns recombinant β-l...,True


In [38]:
comparison[comparison["correct"] == False]

,id,title,abstract,pillar,subpillar,run,subpillar_LLM,status,reasoning_LLM,correct
34,US-20250134149-A1,OGATAEA POLYMORPHA DERIVED COMPOSITIONS AND AP...,The disclosure provided herein is related to t...,F,PF,1,BF,ok,The invention uses methylotrophic yeast (Ogata...,False
44,WO-2025078458-A3,NOVEL FOOD COMPOSITION AND METHOD FOR PRODUCTI...,The present disclosure relates to a process fo...,F,PF,1,NA,ok,The patent describes enzymatic hydrolysis of l...,False
52,DE-102023125143-A1,Container for the fermentation of biomass,The present invention relates to a container f...,F,BF,1,NA,ok,This patent describes a fermentation container...,False


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign scope and pillar, I need to save the comparison data, then manually review what went wrong and adjust the prompt.
None of this will make it into the final workflow.

Order of working:
1. Create a new version folder in the 1_prompt_debugging folder.
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results. Includes both metrics and 
6. Write a text document about v1 results and what changes you want to make to the prompt. Repeat from step 1.

In [39]:
save_dir = Path(PROMPT_VERSION)
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "accuracy", "value": f"{comparison['correct'].mean():.0%}", "n": n},
])

out_path = save_dir / f"{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    confusion.to_excel(             writer, sheet_name="confusion")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")

Saved to v2\v2_claude-sonnet-4-6_results.xlsx


In [40]:
# Records where subpillar_LLM did not match subpillar — for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["correct"], "id"]
incorrect_subpillar_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_subpillar_data

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,US-20250134149-A1,88412235,US18722004,OGATAEA POLYMORPHA DERIVED COMPOSITIONS AND AP...,The disclosure provided herein is related to t...,"['C07K14/805', 'C12N1/16', 'C07K14/39', 'A61K2...",2025,US,in,F,PF,Strain development,Meat,Flavours and aromas,False
1,WO-2025078458-A3,88373901,EP2024/078433,NOVEL FOOD COMPOSITION AND METHOD FOR PRODUCTI...,The present disclosure relates to a process fo...,"['C12Y302/01', 'C12C5/006', 'A23G1/34', 'C12P1...",2026,WO,in,F,PF,Ingredient optimisation,Milk and milk proteins,"Emulsions, gels, and binders",False
2,DE-102023125143-A1,94776239,DE102023125143.0,Container for the fermentation of biomass,The present invention relates to a container f...,"['C12M27/02', 'C12M41/40', 'C12M37/00', 'C12N1...",2025,DE,in,F,BF,Bioprocess design,Meat,NaN,False
